Before data reaches your gold layer or dashboards, it must pass through a data quality gate - 

-  No nulls in critical columns
-  No duplicate records
-  Consistent data types
-  Proper formatting and case normalization

These "small" fixes ensure reliable analytics and downstream ML models.

### 1. Load a Sample Dataset with Issues
We'll intentionally create some dirty data for demonstration.

In [0]:
import pandas as pd
import numpy as np

employees_df = pd.DataFrame({
    "emp_id": [1, 2, 3, 4, 5, 5, 6],
    "emp_name": ["Akash", "Priya", "Rohan", np.nan, "Neha", "Neha", "Raj"],
    "department_id": [10, 20, 30, 10, np.nan, np.nan, 30],
    "salary": [12000, 8000, None, 11000, 9000, 9000, "17000"],
    "joining_date": ["2023-01-01", "2023-03-10", "2023/02/01", None, "2023-03-25", "2023-03-25", "2023-05-15"]
})

department_df = pd.DataFrame({
    "department_id": [10, 20, 30],
    "department_name": ["Sales", "Marketing", "IT"]
})

### 2. Identifying Null Values

In [0]:
print(employees_df.isnull().sum())

In [0]:
import seaborn as sns
sns.heatmap(employees_df.isnull(), cbar=False)

### 3. Handling Missing Data

a) Drop rows with nulls (simple approach)

In [0]:
clean_df = employees_df.dropna()
print(clean_df)

b) Drop nulls only for specific columns

In [0]:
clean_df = employees_df.dropna(subset=["emp_name", "salary"])

c) Fill nulls with default or computed values

In [0]:
employees_df["department_id"].fillna(0, inplace=True)
employees_df["salary"].fillna(employees_df["salary"].median(), inplace=True)

### d) Fill missing names

In [0]:
employees_df["emp_name"].fillna("Unknown", inplace=True)

### 4. Deduplication - Removing Duplicate Records

a) Identify duplicates

In [0]:
duplicates = employees_df[employees_df.duplicated()]
print(duplicates)

b) Drop duplicates

In [0]:
employees_df = employees_df.drop_duplicates()

### 5. Data Type Conversions

Let's fix mixed data types (salary as string, inconsistent dates).

a) Convert salary to numeric

In [0]:
employees_df["salary"] = pd.to_numeric(employees_df["salary"], errors="coerce")
print(employees_df["salary"])

b) Convert joining_date to datetime

In [0]:
employees_df["joining_date"] = pd.to_datetime(employees_df["joining_date"], errors="coerce", infer_datetime_format=True)

c) Replace invalid dates or salary with defaults

In [0]:
employees_df["joining_date"].fillna(pd.Timestamp("2023-01-01"), inplace=True)

### 6. Standardizing Text & Casing

In [0]:
employees_df["emp_name"] = employees_df["emp_name"].str.strip().str.title()
print(employees_df["emp_name"].head())

### 7. Detecting & Handling Outliers (Bonus)
Let's say salaries above 50,000 are unrealistic in this sample.

In [0]:
outliers = employees_df[employees_df["salary"] > 50000]
print(outliers)

### 8. Joining Cleaned Data with Department Info
After cleaning, join to enrich dataset.

In [0]:
final_df = employees_df.merge(department_df, on="department_id", how="left")
print(final_df)

In [0]:
final_df.to_csv("/Volumes/thedataengineering_00/data/sampledata/employees_cleaned.csv", index=False)

### 10. Real-World Data Cleaning Pattern (ETL Style)

In [0]:
def clean_employees(df):
    df["emp_name"] = df["emp_name"].fillna("Unknown").str.strip().str.title()
    df["salary"] = pd.to_numeric(df["salary"], errors="coerce").fillna(df["salary"].median())
    df["department_id"] = df["department_id"].fillna(0)
    df["joining_date"] = pd.to_datetime(df["joining_date"], errors="coerce").fillna(pd.Timestamp("2023-01-01"))
    df = df.drop_duplicates(subset=["emp_id"])
    return df

employees_df = clean_employees(employees_df)
final_df = employees_df.merge(department_df, on="department_id", how="left")

Practice Tasks

- Identify missing values per column in employees_df.
-  Drop rows with null employee names.
-  Convert salary to numeric and fill missing with median.
-  Remove duplicate employee IDs.
-  Normalize text columns (names, departments).
-  Write the cleaned dataset to employees_cleaned.csv.